# Patch-wise PCA for Industrial Anomaly Detection

**CS 663 — Digital Image Processing | MVTec AD**

This notebook implements patch-wise Principal Component Analysis using NumPy linear algebra. It does not use a ready-made PCA class.

Instead of representing an entire image with one long vector, we split each normal training image into overlapping local patches. PCA learns common local appearances from these patches. At test time, every patch is reconstructed and its pixel-wise reconstruction error is placed back into the original image. Overlapping error maps are averaged.

Only normal training images are used to fit the model. Validation images set the image-level threshold. The official test split is used only for final evaluation.

## 1. Setup and data loading

Run this notebook from the repository in Colab or from a local clone. The default category is bottle. Change CATEGORY to evaluate another category.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from scipy.ndimage import label as connected_components

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/CS-663")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/pranitas-cloud/CS-663.git", str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / "src").is_dir():
        raise RuntimeError("Open this notebook from the CS-663 repository root.")

os.chdir(REPO)
if str(REPO.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO.resolve()))

from src.data.pipelines import load_config, build_dataset, stack_images

CONFIG = load_config("configs/default.yaml")
DATA_ROOT = Path(os.environ.get("MVTEC_DATA_ROOT", REPO / "data" / "mvtec_anomaly_detection")).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO / DATA_ROOT
CONFIG["data"]["root"] = str(DATA_ROOT)

CATEGORY = "bottle"
IMAGE_SIZE = int(CONFIG["pca"].get("image_size", 256))
SEED = int(CONFIG.get("seed", 42))
np.random.seed(SEED)
torch.manual_seed(SEED)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"MVTec folder not found: {DATA_ROOT}")

print("Repository:", REPO)
print("Dataset:", DATA_ROOT)
print("Category:", CATEGORY)
print("Image size:", IMAGE_SIZE)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

## 2. Patch-wise PCA implementation

For every training patch, we subtract the mean patch and calculate the covariance matrix. The eigenvectors with the largest eigenvalues form the retained principal subspace.

The patch size and stride determine how many patches are extracted. A fixed random sample of patches is used to keep memory use manageable.

In [ ]:
class PatchPCA:
    def __init__(self, patch_size=32, stride=16, retained_variance=0.95,
                 maximum_patches=10000, random_seed=42):
        if patch_size < 1 or stride < 1:
            raise ValueError("patch_size and stride must be positive.")
        if not 0.0 < retained_variance <= 1.0:
            raise ValueError("retained_variance must be between 0 and 1.")
        self.patch_size = int(patch_size)
        self.stride = int(stride)
        self.retained_variance = retained_variance
        self.maximum_patches = int(maximum_patches)
        self.random_seed = int(random_seed)
        self.mean = None
        self.components = None
        self.explained_variance = None
        self.image_shape = None
        self.patch_positions = None
        self.patches_available = 0
        self.patches_used = 0

    def get_positions(self, height, width):
        patch_size = self.patch_size
        stride = self.stride

        if height < patch_size or width < patch_size:
            raise ValueError("Patch size is larger than the image.")
        if (height - patch_size) % stride != 0:
            raise ValueError("Image height must be covered exactly by the selected stride.")
        if (width - patch_size) % stride != 0:
            raise ValueError("Image width must be covered exactly by the selected stride.")

        positions = []
        for top in range(0, height - patch_size + 1, stride):
            for left in range(0, width - patch_size + 1, stride):
                positions.append((top, left))
        return positions

    def fit(self, training_images):
        images = np.asarray(training_images, dtype=np.float32)

        if images.ndim != 4:
            raise ValueError("Images must have shape (N, C, H, W).")
        if not np.isfinite(images).all():
            raise ValueError("Training images contain invalid values.")

        number_of_images, channels, height, width = images.shape
        self.image_shape = (channels, height, width)
        self.patch_positions = self.get_positions(height, width)

        all_positions = []
        for image_number in range(number_of_images):
            for top, left in self.patch_positions:
                all_positions.append((image_number, top, left))

        if self.maximum_patches < 2:
            raise ValueError("maximum_patches must be at least 2.")

        self.patches_available = len(all_positions)
        number_to_use = min(self.maximum_patches, self.patches_available)
        self.patches_used = number_to_use
        generator = np.random.default_rng(self.random_seed)
        chosen_positions = generator.choice(len(all_positions), size=number_to_use, replace=False)

        patch_dimension = channels * self.patch_size * self.patch_size
        training_patches = np.empty((number_to_use, patch_dimension), dtype=np.float64)

        for row, position_index in enumerate(chosen_positions):
            image_number, top, left = all_positions[position_index]
            patch = images[
                image_number,
                :,
                top:top + self.patch_size,
                left:left + self.patch_size
            ]
            training_patches[row] = patch.reshape(-1)

        self.mean = training_patches.mean(axis=0)
        centered_patches = training_patches - self.mean
        covariance = centered_patches.T @ centered_patches / max(number_to_use - 1, 1)

        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = np.maximum(eigenvalues[order], 0.0)
        eigenvectors = eigenvectors[:, order]

        valid = eigenvalues > eigenvalues[0] * 1e-10
        eigenvalues = eigenvalues[valid]
        eigenvectors = eigenvectors[:, valid]

        if len(eigenvalues) == 0:
            raise ValueError("The training patches do not contain measurable variance.")

        variance_ratios = eigenvalues / eigenvalues.sum()
        component_count = np.searchsorted(
            np.cumsum(variance_ratios), self.retained_variance
        ) + 1

        self.components = eigenvectors[:, :component_count].T
        self.explained_variance = variance_ratios[:component_count]
        return self

    def transform_patch(self, patch):
        centered_patch = patch - self.mean
        reduced_patch = centered_patch @ self.components.T
        return reduced_patch @ self.components + self.mean

    def reconstruct(self, images):
        if self.components is None:
            raise RuntimeError("Fit the model before reconstruction.")

        images = np.asarray(images, dtype=np.float32)
        if images.shape[1:] != self.image_shape:
            raise ValueError("The input image dimensions do not match the training images.")

        number_of_images, channels, height, width = images.shape
        patch_size = self.patch_size
        reconstructed_images = np.zeros_like(images, dtype=np.float64)
        overlap_count = np.zeros((height, width), dtype=np.float64)

        for top, left in self.patch_positions:
            overlap_count[
                top:top + patch_size,
                left:left + patch_size
            ] += 1

        for image_number in range(number_of_images):
            for top, left in self.patch_positions:
                patch = images[
                    image_number,
                    :,
                    top:top + patch_size,
                    left:left + patch_size
                ]
                restored_patch = self.transform_patch(patch.reshape(-1))
                restored_patch = restored_patch.reshape(channels, patch_size, patch_size)
                reconstructed_images[
                    image_number,
                    :,
                    top:top + patch_size,
                    left:left + patch_size
                ] += restored_patch

        reconstructed_images /= overlap_count[None, None, :, :]
        return reconstructed_images.astype(np.float32)

    def anomaly_map(self, images):
        images = np.asarray(images, dtype=np.float32)

        if images.shape[1:] != self.image_shape:
            raise ValueError("The input image dimensions do not match the training images.")

        number_of_images, channels, height, width = images.shape
        patch_size = self.patch_size
        anomaly_maps = np.zeros((number_of_images, height, width), dtype=np.float64)
        overlap_count = np.zeros((height, width), dtype=np.float64)

        for top, left in self.patch_positions:
            overlap_count[
                top:top + patch_size,
                left:left + patch_size
            ] += 1

        for image_number in range(number_of_images):
            for top, left in self.patch_positions:
                patch = images[
                    image_number,
                    :,
                    top:top + patch_size,
                    left:left + patch_size
                ]
                restored_patch = self.transform_patch(patch.reshape(-1))
                restored_patch = restored_patch.reshape(channels, patch_size, patch_size)
                patch_error = ((patch - restored_patch) ** 2).mean(axis=0)

                anomaly_maps[
                    image_number,
                    top:top + patch_size,
                    left:left + patch_size
                ] += patch_error

        anomaly_maps /= overlap_count[None, :, :]
        return anomaly_maps.astype(np.float32)

    def save(self, file_path):
        if self.components is None:
            raise RuntimeError("Fit the model before saving.")

        np.savez_compressed(
            file_path,
            mean=self.mean,
            components=self.components,
            explained_variance=self.explained_variance,
            image_shape=np.asarray(self.image_shape),
            patch_size=self.patch_size,
            stride=self.stride
        )

## 3. Check patch extraction and reconstruction

The first check verifies that extracting and averaging overlapping patches reproduces the original image. The second check verifies that the PCA model produces finite anomaly maps with the expected dimensions.

In [ ]:
toy_image = np.random.default_rng(SEED).normal(size=(5, 1, 32, 32)).astype(np.float32)
toy_model = PatchPCA(
    patch_size=8,
    stride=4,
    retained_variance=1.0,
    maximum_patches=500,
    random_seed=SEED
).fit(toy_image)

toy_reconstruction = toy_model.reconstruct(toy_image)
toy_anomaly_map = toy_model.anomaly_map(toy_image)

assert toy_reconstruction.shape == toy_image.shape
assert toy_anomaly_map.shape == (5, 32, 32)
assert np.isfinite(toy_anomaly_map).all()
assert toy_anomaly_map.min() >= 0

print("Patch PCA synthetic checks passed.")

## 4. Load the MVTec splits

The validation split is created from normal training images. We check this before extracting patches or fitting PCA.

In [ ]:
train_dataset = build_dataset(CONFIG, "pca", "train", CATEGORY)
validation_dataset = build_dataset(CONFIG, "pca", "val", CATEGORY)
test_dataset = build_dataset(CONFIG, "pca", "test", CATEGORY)

training_images = stack_images(train_dataset).cpu().numpy()

assert all(sample[2] == "good" for sample in train_dataset.samples)
assert all(sample[2] == "good" for sample in validation_dataset.samples)

print("Training images:", training_images.shape)
print("Validation images:", len(validation_dataset))
print("Test images:", len(test_dataset))

## 5. Fit patch-wise PCA

The baseline uses 32 × 32 patches, stride 16, 95% retained variance and at most 10,000 training patches. These settings can be tuned using validation performance.

In [ ]:
PATCH_SIZE = int(CONFIG["pca"].get("patch_size", 32))
PATCH_STRIDE = int(CONFIG["pca"].get("stride", 16))
RETAINED_VARIANCE = 0.95
MAXIMUM_TRAINING_PATCHES = 10000

patch_model = PatchPCA(
    patch_size=PATCH_SIZE,
    stride=PATCH_STRIDE,
    retained_variance=RETAINED_VARIANCE,
    maximum_patches=MAXIMUM_TRAINING_PATCHES,
    random_seed=SEED
)

patch_model.fit(training_images)

print("Number of principal components:", len(patch_model.components))
print("Retained training variance:", patch_model.explained_variance.sum())
print("Training patches available:", patch_model.patches_available)
print("Training patches used:", patch_model.patches_used)

## 6. Visualize a normal validation image

Each local patch is reconstructed independently. The final reconstruction averages predictions in regions covered by multiple patches.

In [ ]:
example = validation_dataset[0]
example_image = example["image"].unsqueeze(0).cpu().numpy()

original_image = example_image[0, 0]
reconstructed_image = patch_model.reconstruct(example_image)[0, 0]
error_map = patch_model.anomaly_map(example_image)[0]

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(original_image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(reconstructed_image, cmap="gray")
axes[1].set_title("Patch PCA reconstruction")
axes[2].imshow(error_map, cmap="inferno")
axes[2].set_title("Anomaly map")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. Evaluation

The image threshold is set using only normal validation images, at the 5% false-positive target. It is fixed before evaluating the official test split.

The evaluation reports image AUROC and AP, image precision/recall/F1 at the validation threshold, pixel AUROC and AP, and normalized AUPRO up to 30% pixel false-positive rate.

In [ ]:
def collect_outputs(dataset, model):
    image_labels = []
    image_scores = []
    masks = []
    anomaly_maps = []

    for index in range(len(dataset)):
        item = dataset[index]
        image = item["image"].unsqueeze(0).cpu().numpy()
        anomaly_map = model.anomaly_map(image)[0]
        mask = item["mask"].squeeze(0).cpu().numpy().astype(bool)

        image_labels.append(int(item["label"]))
        image_scores.append(float(anomaly_map.max()))
        masks.append(mask)
        anomaly_maps.append(anomaly_map)

    return (
        np.asarray(image_labels, dtype=np.uint8),
        np.asarray(image_scores, dtype=np.float64),
        np.stack(masks),
        np.stack(anomaly_maps)
    )


def compute_aupro(masks, anomaly_maps, max_fpr=0.30, number_of_thresholds=128):
    masks = np.asarray(masks, dtype=bool)
    anomaly_maps = np.asarray(anomaly_maps, dtype=np.float64)

    regions = []
    for mask in masks:
        region_labels, number_of_regions = connected_components(mask)
        for region_number in range(1, number_of_regions + 1):
            region = region_labels == region_number
            if region.any():
                regions.append(region)

    negative_pixels = int((~masks).sum())
    if not regions or negative_pixels == 0:
        return float("nan")

    thresholds = np.unique(
        np.quantile(anomaly_maps, np.linspace(0.0, 1.0, number_of_thresholds))
    )[::-1]

    false_positive_rates = [0.0]
    region_overlaps = [0.0]

    for threshold in thresholds:
        prediction = anomaly_maps >= threshold
        false_positive_rate = np.logical_and(prediction, ~masks).sum() / negative_pixels

        if false_positive_rate > max_fpr:
            break

        overlaps = []
        for region in regions:
            overlaps.append(np.logical_and(prediction, region).sum() / region.sum())

        false_positive_rates.append(float(false_positive_rate))
        region_overlaps.append(float(np.mean(overlaps)))

    x_values = np.asarray(false_positive_rates)
    y_values = np.asarray(region_overlaps)
    order = np.argsort(x_values)
    x_values = x_values[order]
    y_values = y_values[order]

    unique_x = np.unique(x_values)
    unique_y = np.asarray([y_values[x_values == value].max() for value in unique_x])

    inside = unique_x < max_fpr
    x_values = unique_x[inside]
    y_values = unique_y[inside]

    if len(x_values) == 0 or x_values[0] != 0.0:
        x_values = np.insert(x_values, 0, 0.0)
        y_values = np.insert(y_values, 0, 0.0)

    end_value = float(np.interp(max_fpr, unique_x, unique_y))
    x_values = np.append(x_values, max_fpr)
    y_values = np.append(y_values, end_value)

    return float(np.clip(np.trapz(y_values, x_values) / max_fpr, 0.0, 1.0))


def evaluate_model(model, validation_dataset, test_dataset, model_name):
    validation_labels, validation_scores, _, _ = collect_outputs(validation_dataset, model)
    if np.any(validation_labels != 0):
        raise ValueError("Validation data must contain normal images only.")

    image_threshold = float(np.quantile(validation_scores, 0.95, method="higher"))

    test_labels, test_scores, test_masks, test_maps = collect_outputs(test_dataset, model)
    if np.unique(test_labels).size != 2:
        raise ValueError("The test split must contain both normal and anomalous images.")

    image_predictions = (test_scores >= image_threshold).astype(np.uint8)
    flat_masks = test_masks.reshape(-1).astype(np.uint8)
    flat_scores = test_maps.reshape(-1)

    results = {
        "category": CATEGORY,
        "method": model_name,
        "image_auroc": float(roc_auc_score(test_labels, test_scores)),
        "image_ap": float(average_precision_score(test_labels, test_scores)),
        "image_precision": float(precision_score(test_labels, image_predictions, zero_division=0)),
        "image_recall": float(recall_score(test_labels, image_predictions, zero_division=0)),
        "image_f1": float(f1_score(test_labels, image_predictions, zero_division=0)),
        "pixel_auroc": float(roc_auc_score(flat_masks, flat_scores)),
        "pixel_ap": float(average_precision_score(flat_masks, flat_scores)),
        "aupro_30pct_fpr": compute_aupro(test_masks, test_maps),
        "validation_threshold": image_threshold
    }

    return pd.DataFrame([results]), test_maps, test_masks

patch_metrics, patch_test_maps, patch_test_masks = evaluate_model(
    patch_model,
    validation_dataset,
    test_dataset,
    "Patch-wise PCA"
)

display(patch_metrics.round(4))

## 8. Save the model and results

In [ ]:
OUTPUT_DIR = Path("experiments/pca")
RESULTS_DIR = Path("results/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / f"{CATEGORY}_patch_pca.npz"
metrics_path = RESULTS_DIR / f"{CATEGORY}_patch_pca_metrics.csv"

patch_model.save(model_path)
patch_metrics.to_csv(metrics_path, index=False)

print("Model saved to:", model_path)
print("Metrics saved to:", metrics_path)

## 9. Interpretation and limitations

- Patch-wise PCA focuses on local texture and structure, so it can localize defects more precisely than a single global subspace.
- Overlapping patches reduce abrupt boundaries between adjacent patch predictions.
- Reconstruction error is an anomaly score, not a probability.
- Patch size, stride, retained variance and the number of sampled patches should be compared using validation data only.
- Results in this notebook cover the selected category only. Repeat the same protocol for the other MVTec categories before making dataset-wide claims.
- The detector is fitted only on normal images. Defect labels and masks are used only for final evaluation.